# FaunaVision 01: camera-trap crops + frozen DINOv2 baseline

Runs on a free Kaggle GPU notebook (**Settings: Accelerator = GPU T4 x2 or P100, Internet = On**).

1. Rebuild the camera-trap index and manifest from LILA metadata (~10 min)
2. Download and crop the 154k manifest images (~1–2 h, resumable)
3. Extract DINOv2 ViT-B/14 features (~20 min)
4. Linear probe + k-NN baseline, results in `reports/results/`

When it finishes, **Save Version → Save & Run All**, then use *Output → New Dataset* on `/kaggle/working/crops`
so later notebooks reuse the crops instead of downloading again.

In [ ]:
REPO_URL = "https://github.com/<your-username>/faunavision.git"  # set this
!git clone -q $REPO_URL /kaggle/working/faunavision
%cd /kaggle/working/faunavision
!pip install -q ijson timm

## 1. Metadata, index and manifest

In [ ]:
!mkdir -p data/raw/lila
!wget -q -O data/raw/lila/ss2024_meta.zip https://storage.googleapis.com/public-datasets-lila/snapshot-safari-2024-expansion/snapshot_safari_2024_metadata.zip
!wget -q -O data/raw/lila/md_ss2024_rde.json.zip https://lila.science/public/lila-md-results/snapshot_safari_2024_lila_mdv5a-rde_0.100_0.850_15_0.200.json.zip
!python scripts/build_camtrap_index.py
!python scripts/make_manifest.py --test-projects PLN,KRU,CDB

## 2. Download crops (re-run this cell if the session drops: finished crops are skipped)

In [ ]:
!python scripts/download_crops.py --out /kaggle/working/crops --workers 32

## 3. Frozen DINOv2 features

In [ ]:
!python scripts/extract_features.py --crops /kaggle/working/crops --out /kaggle/working/features --batch-size 256

## 4. Linear probe and k-NN

In [ ]:
!python scripts/linear_probe.py --features /kaggle/working/features --run dinov2_vitb14_probe
from IPython.display import Markdown
Markdown(open("reports/results/dinov2_vitb14_probe/report.md").read())